# AEGIS-NET: CIC-IDS-2017 training and artifact export

This notebook creates a **new** training pipeline from the public Kaggle dataset [`chethuhn/network-intrusion-dataset`](https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset). It does not claim to reproduce any earlier Colab model or its preprocessing. The feature selection, imputation values, scaling parameters, label order, model configurations, and anomaly threshold are fitted or selected here and exported with the models. The application must consume those exported values rather than infer new preprocessing.

Run this notebook in Google Colab. It downloads the dataset into the Colab runtime, samples each class with a bounded reservoir, trains a multiclass XGBoost classifier, a PyTorch MLP, and a benign-only PyTorch autoencoder, then creates a zip containing **only model and preprocessing artifacts**. Evaluation values are generated by the run; none are prefilled. The split is a random in-dataset split, so metrics are not a guarantee of performance on new networks or future traffic. Review dataset provenance and validate models before operational use.

If Kaggle requests authentication, configure credentials using Colab Secrets or Kaggle's supported authentication flow. Do not paste API tokens into this notebook or commit them to Git.

In [1]:
%pip -q install kagglehub xgboost scikit-learn

import datetime as dt
import json
import platform
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

SEED = 42
CHUNK_ROWS = 100_000
MAX_ROWS_PER_CLASS = 25_000
TEST_FRACTION = 0.20
BATCH_SIZE = 2_048
CLASSIFIER_EPOCHS = 10
AUTOENCODER_EPOCHS = 10
ANOMALY_QUANTILE = 0.995

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('XGBoost:', xgb.__version__)
print('Training device:', DEVICE)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\diksh\OneDrive\Desktop\DL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pandas'

## Download and bounded, class-stratified sampling

CSV headers are normalized by trimming whitespace and a UTF-8 BOM. The target must be a single `Label` or `Class` column. Flow IDs, IP addresses, timestamps, and target columns are excluded from model inputs. Remaining columns are numeric-coerced; nonnumeric/all-missing columns are omitted from the final feature list. A deterministic reservoir keeps at most `MAX_ROWS_PER_CLASS` rows per label while streaming chunks.

In [ ]:
DATASET_ID = 'chethuhn/network-intrusion-dataset'
DATA_ROOT = Path(kagglehub.dataset_download(DATASET_ID))
CSV_FILES = sorted(DATA_ROOT.rglob('*.csv'))
if not CSV_FILES:
    raise FileNotFoundError(f'No CSV files found under {DATA_ROOT}')
print(f'Found {len(CSV_FILES)} CSV files under {DATA_ROOT}')

LABEL_NAMES = {'label', 'class'}
METADATA_NAMES = {
    'flow id', 'source ip', 'destination ip', 'src ip', 'dst ip',
    'source address', 'destination address', 'timestamp'
}

def normalize_column(name):
    return ' '.join(str(name).replace(chr(65279), '').strip().split())

def update_reservoir(label, new_rows, reservoirs, seen_counts, rng):
    previous_count = seen_counts.get(label, 0)
    current = reservoirs.get(label)
    if current is None:
        current = np.empty((0, new_rows.shape[1]), dtype=np.float32)

    add_count = min(max(MAX_ROWS_PER_CLASS - previous_count, 0), len(new_rows))
    if add_count > 0:
        current = np.concatenate((current, new_rows[:add_count]), axis=0)

    remaining = new_rows[add_count:]
    if len(remaining):
        row_numbers = previous_count + np.arange(add_count + 1, len(new_rows) + 1)
        slots = (rng.random(len(remaining)) * row_numbers).astype(np.int64)
        replace = slots < MAX_ROWS_PER_CLASS
        for slot, row in zip(slots[replace], remaining[replace]):
            current[slot] = row

    reservoirs[label] = current
    seen_counts[label] = previous_count + len(new_rows)

rng = np.random.default_rng(SEED)
reservoirs = {}
seen_counts = {}
expected_candidates = None

for csv_path in CSV_FILES:
    print('Reading:', csv_path.name)
    for chunk in pd.read_csv(
        csv_path,
        chunksize=CHUNK_ROWS,
        encoding='utf-8-sig',
        encoding_errors='replace',
        low_memory=False,
    ):
        chunk.columns = [normalize_column(name) for name in chunk.columns]
        if len(set(chunk.columns)) != len(chunk.columns):
            raise ValueError(f'Duplicate columns after header normalization in {csv_path.name}')

        label_columns = [name for name in chunk.columns if name.casefold() in LABEL_NAMES]
        if len(label_columns) != 1:
            raise ValueError(f'Expected exactly one Label/Class column in {csv_path.name}; found {label_columns}')
        label_column = label_columns[0]
        candidates = [
            name for name in chunk.columns
            if name != label_column
            and name.casefold() not in LABEL_NAMES
            and name.casefold() not in METADATA_NAMES
        ]
        if expected_candidates is None:
            expected_candidates = candidates
        elif set(candidates) != set(expected_candidates):
            raise ValueError(f'Feature schema differs in {csv_path.name}; refusing to silently reorder or invent features')

        raw_labels = chunk[label_column].astype('string').str.strip()
        valid_labels = raw_labels.notna() & raw_labels.ne('')
        if not valid_labels.any():
            continue

        values = chunk.loc[valid_labels, expected_candidates].apply(
            pd.to_numeric, errors='coerce'
        ).to_numpy(dtype=np.float32)
        values[~np.isfinite(values)] = np.nan
        labels = raw_labels.loc[valid_labels].astype(str).to_numpy()
        for label in np.unique(labels):
            update_reservoir(label, values[labels == label], reservoirs, seen_counts, rng)

if not reservoirs:
    raise ValueError('No labeled flow rows were read from the dataset.')

class_names = sorted(reservoirs)
raw_features = np.concatenate([reservoirs[name] for name in class_names], axis=0)
raw_labels = np.concatenate([
    np.full(len(reservoirs[name]), name, dtype=object) for name in class_names
])
usable_columns = np.isfinite(raw_features).any(axis=0)
feature_order = [name for name, usable in zip(expected_candidates, usable_columns) if usable]
raw_features = raw_features[:, usable_columns]
if not feature_order:
    raise ValueError('No numeric flow features remained after excluding metadata and empty columns.')

print('Retained features:', len(feature_order))
for name in class_names:
    print(f'{name}: sampled={len(reservoirs[name])}, labeled_rows_scanned={seen_counts[name]}')

## Fit preprocessing on training rows only

The median-imputation and population-standardization parameters below are fit only on the training partition. They are exported in the same order as `feature_order`; an inference application must use these exact arrays. The random split is stratified by label. Labels with only one sampled row remain in training and are reported as absent from the evaluation partition.

In [ ]:
class_to_index = {name: index for index, name in enumerate(class_names)}
y = np.asarray([class_to_index[name] for name in raw_labels], dtype=np.int64)
split_rng = np.random.default_rng(SEED + 1)
train_parts = []
test_parts = []
single_row_classes = []

for class_index, class_name in enumerate(class_names):
    indices = np.flatnonzero(y == class_index)
    split_rng.shuffle(indices)
    if len(indices) < 2:
        train_parts.append(indices)
        single_row_classes.append(class_name)
        continue
    test_count = max(1, int(round(len(indices) * TEST_FRACTION)))
    test_count = min(test_count, len(indices) - 1)
    test_parts.append(indices[:test_count])
    train_parts.append(indices[test_count:])

train_indices = np.concatenate(train_parts)
if not test_parts:
    raise ValueError('The sampled data has no class with enough rows for an evaluation split.')
test_indices = np.concatenate(test_parts)
if single_row_classes:
    print('Classes retained only in training (one sampled row):', single_row_classes)

X_train_raw = raw_features[train_indices].astype(np.float64)
X_test_raw = raw_features[test_indices].astype(np.float64)
y_train = y[train_indices]
y_test = y[test_indices]

imputation_medians = np.nanmedian(X_train_raw, axis=0)
if not np.isfinite(imputation_medians).all():
    raise ValueError('A retained feature has no finite training values; review the dataset schema.')
X_train_imputed = np.where(np.isfinite(X_train_raw), X_train_raw, imputation_medians)
X_test_imputed = np.where(np.isfinite(X_test_raw), X_test_raw, imputation_medians)
scaler_means = X_train_imputed.mean(axis=0)
scaler_scales = X_train_imputed.std(axis=0, ddof=0)
scaler_scales[~np.isfinite(scaler_scales) | (scaler_scales == 0)] = 1.0
X_train = ((X_train_imputed - scaler_means) / scaler_scales).astype(np.float32)
X_test = ((X_test_imputed - scaler_means) / scaler_scales).astype(np.float32)

print('Training rows:', len(train_indices))
print('Evaluation rows:', len(test_indices))
print('Features:', len(feature_order))
print('Classes:', class_names)

## Train and measure the supervised classifiers

XGBoost and the MLP use the same fitted feature transformation and label order. Their reported scores are model outputs, not guarantees of correctness or calibrated probabilities. The MLP architecture configuration is saved alongside its state dictionary so the application can reconstruct the exact layer layout.

In [ ]:
def measure_classifier(name, predictions):
    metrics = {
        'accuracy': float(accuracy_score(y_test, predictions)),
        'macro_f1': float(f1_score(
            y_test, predictions, labels=list(range(len(class_names))),
            average='macro', zero_division=0
        )),
    }
    print(f'--- {name} ---')
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        y_test,
        predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,
    ))
    return metrics

xgb_classifier = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(class_names),
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    eval_metric='mlogloss',
    n_jobs=4,
    random_state=SEED,
)
xgb_classifier.fit(X_train, y_train)
xgb_metrics = measure_classifier('XGBoost', xgb_classifier.predict(X_test))

mlp_config = {
    'architecture': 'dense_mlp_v1',
    'input_size': len(feature_order),
    'num_classes': len(class_names),
    'hidden_sizes': [128, 64],
    'dropout': 0.1,
}

def build_mlp(config):
    sizes = [config['input_size']] + config['hidden_sizes']
    layers = []
    for index in range(len(config['hidden_sizes'])):
        layers.extend([nn.Linear(sizes[index], sizes[index + 1]), nn.ReLU()])
        if index == 0 and config['dropout'] > 0:
            layers.append(nn.Dropout(config['dropout']))
    layers.append(nn.Linear(config['hidden_sizes'][-1], config['num_classes']))
    return nn.Sequential(*layers)

mlp = build_mlp(mlp_config).to(DEVICE)
mlp_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
mlp_optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3)
mlp_loss_fn = nn.CrossEntropyLoss()
for epoch in range(CLASSIFIER_EPOCHS):
    mlp.train()
    total_loss = 0.0
    for batch_x, batch_y in mlp_loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        mlp_optimizer.zero_grad(set_to_none=True)
        loss = mlp_loss_fn(mlp(batch_x), batch_y)
        loss.backward()
        mlp_optimizer.step()
        total_loss += loss.item() * len(batch_x)
    print(f'MLP epoch {epoch + 1}/{CLASSIFIER_EPOCHS}: loss={total_loss / len(X_train):.6f}')

mlp.eval()
with torch.no_grad():
    mlp_logits = mlp(torch.from_numpy(X_test).to(DEVICE))
    mlp_predictions = mlp_logits.argmax(dim=1).cpu().numpy()
mlp_metrics = measure_classifier('PyTorch MLP', mlp_predictions)

## Train and calibrate the benign-only autoencoder

The autoencoder is trained only on training-partition rows labeled `BENIGN` or `Normal`. Its reconstruction-MSE threshold is the configured quantile of a separate benign calibration subset. This threshold is a measured calibration value, not a universal anomaly boundary.

In [ ]:
normal_class_ids = [
    index for index, name in enumerate(class_names)
    if name.strip().casefold() in {'benign', 'normal'}
]
if not normal_class_ids:
    raise ValueError(f'No BENIGN/Normal class found. Observed labels: {class_names}')

normal_train = X_train[np.isin(y_train, normal_class_ids)]
if len(normal_train) < 20:
    raise ValueError(f'At least 20 benign training rows are required; found {len(normal_train)}')
ae_rng = np.random.default_rng(SEED + 2)
normal_order = ae_rng.permutation(len(normal_train))
calibration_size = max(10, int(round(len(normal_train) * 0.20)))
if calibration_size >= len(normal_train):
    raise ValueError('Not enough benign training rows to separate fit and calibration subsets.')
calibration_indices = normal_order[:calibration_size]
ae_fit_indices = normal_order[calibration_size:]
ae_fit = normal_train[ae_fit_indices]
ae_calibration = normal_train[calibration_indices]

autoencoder_config = {
    'architecture': 'dense_autoencoder_v1',
    'input_size': len(feature_order),
    'layer_sizes': [len(feature_order), 128, 64, 32, 64, 128, len(feature_order)],
    'activation': 'relu',
}

def build_autoencoder(config):
    sizes = config['layer_sizes']
    layers = []
    for index in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[index], sizes[index + 1]))
        if index < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

autoencoder = build_autoencoder(autoencoder_config).to(DEVICE)
ae_loader = DataLoader(
    TensorDataset(torch.from_numpy(ae_fit)),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
ae_optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=1e-3)
for epoch in range(AUTOENCODER_EPOCHS):
    autoencoder.train()
    total_loss = 0.0
    for (batch_x,) in ae_loader:
        batch_x = batch_x.to(DEVICE)
        ae_optimizer.zero_grad(set_to_none=True)
        loss = torch.mean((autoencoder(batch_x) - batch_x) ** 2)
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item() * len(batch_x)
    print(f'Autoencoder epoch {epoch + 1}/{AUTOENCODER_EPOCHS}: loss={total_loss / len(ae_fit):.6f}')

def reconstruction_mse(model, values):
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = torch.from_numpy(values[start:start + BATCH_SIZE]).to(DEVICE)
            errors = torch.mean((model(batch) - batch) ** 2, dim=1)
            scores.append(errors.cpu().numpy())
    return np.concatenate(scores)

calibration_errors = reconstruction_mse(autoencoder, ae_calibration)
anomaly_threshold = float(np.quantile(calibration_errors, ANOMALY_QUANTILE))
test_errors = reconstruction_mse(autoencoder, X_test)
true_anomaly = (~np.isin(y_test, normal_class_ids)).astype(np.int32)
predicted_anomaly = (test_errors > anomaly_threshold).astype(np.int32)
anomaly_metrics = {
    'calibration_quantile': ANOMALY_QUANTILE,
    'reconstruction_mse_threshold': anomaly_threshold,
    'test_precision': float(precision_score(true_anomaly, predicted_anomaly, zero_division=0)),
    'test_recall': float(recall_score(true_anomaly, predicted_anomaly, zero_division=0)),
    'test_f1': float(f1_score(true_anomaly, predicted_anomaly, zero_division=0)),
}
print(json.dumps(anomaly_metrics, indent=2))

## Export the versioned artifact bundle

The zip contains portable model files and the exact fitted preprocessing metadata, not the downloaded dataset or raw traffic rows. Once downloaded, place/extract its contents under this project's `artifacts` directory. Do not use the models operationally until the artifact bundle, feature compatibility, and detection behavior have been reviewed.

In [ ]:
EXPORT_ROOT = Path('/content/aegis_artifact_bundle')
CLASSIFIER_DIR = EXPORT_ROOT / 'classifiers'
AUTOENCODER_DIR = EXPORT_ROOT / 'autoencoders'
PREPROCESSING_DIR = EXPORT_ROOT / 'preprocessing'
for directory in (CLASSIFIER_DIR, AUTOENCODER_DIR, PREPROCESSING_DIR):
    directory.mkdir(parents=True, exist_ok=True)

xgb_path = CLASSIFIER_DIR / 'xgboost_classifier.json'
mlp_path = CLASSIFIER_DIR / 'mlp_state_dict.pt'
mlp_config_path = CLASSIFIER_DIR / 'mlp_config.json'
autoencoder_path = AUTOENCODER_DIR / 'autoencoder_state_dict.pt'
autoencoder_config_path = AUTOENCODER_DIR / 'autoencoder_config.json'
preprocessing_path = PREPROCESSING_DIR / 'preprocessing.json'
labels_path = PREPROCESSING_DIR / 'labels.json'
thresholds_path = PREPROCESSING_DIR / 'thresholds.json'

xgb_classifier.save_model(str(xgb_path))
torch.save(mlp.cpu().state_dict(), mlp_path)
torch.save(autoencoder.cpu().state_dict(), autoencoder_path)

def write_json(path, value):
    with path.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2, allow_nan=False)

write_json(mlp_config_path, mlp_config)
write_json(autoencoder_config_path, autoencoder_config)
write_json(preprocessing_path, {
    'schema_version': 1,
    'transform': 'replace non-finite with training median, then (value - mean) / scale',
    'feature_order': feature_order,
    'imputation_medians': imputation_medians.tolist(),
    'scaler_means': scaler_means.tolist(),
    'scaler_scales': scaler_scales.tolist(),
})
write_json(labels_path, {'schema_version': 1, 'classes': class_names})
write_json(thresholds_path, {
    'schema_version': 1,
    'autoencoder_score': 'per-flow mean squared reconstruction error',
    'autoencoder_threshold': anomaly_threshold,
    'calibration_quantile': ANOMALY_QUANTILE,
    'normal_labels': [class_names[index] for index in normal_class_ids],
})

def package_version(distribution):
    from importlib.metadata import PackageNotFoundError, version
    try:
        return version(distribution)
    except PackageNotFoundError:
        return 'not-installed'

metrics = {
    'xgboost': xgb_metrics,
    'mlp': mlp_metrics,
    'autoencoder_anomaly_detection': anomaly_metrics,
    'test_rows': int(len(test_indices)),
    'test_class_support': {
        class_names[index]: int(np.sum(y_test == index))
        for index in range(len(class_names))
    },
}
metrics_path = EXPORT_ROOT / 'evaluation_metrics.json'
write_json(metrics_path, metrics)

artifact_files = {
    'xgboost_classifier': 'classifiers/xgboost_classifier.json',
    'mlp_state_dict': 'classifiers/mlp_state_dict.pt',
    'mlp_config': 'classifiers/mlp_config.json',
    'autoencoder_state_dict': 'autoencoders/autoencoder_state_dict.pt',
    'autoencoder_config': 'autoencoders/autoencoder_config.json',
    'preprocessing': 'preprocessing/preprocessing.json',
    'labels': 'preprocessing/labels.json',
    'thresholds': 'preprocessing/thresholds.json',
    'evaluation_metrics': 'evaluation_metrics.json',
}
manifest = {
    'schema_version': 1,
    'project': 'AEGIS-NET',
    'dataset': DATASET_ID,
    'trained_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'library_versions': {
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit-learn': package_version('scikit-learn'),
        'xgboost': xgb.__version__,
        'torch': torch.__version__,
    },
    'feature_count': len(feature_order),
    'feature_order': feature_order,
    'classes': class_names,
    'training_parameters': {
        'seed': SEED,
        'max_rows_per_class': MAX_ROWS_PER_CLASS,
        'test_fraction': TEST_FRACTION,
        'classifier_epochs': CLASSIFIER_EPOCHS,
        'autoencoder_epochs': AUTOENCODER_EPOCHS,
        'anomaly_quantile': ANOMALY_QUANTILE,
    },
    'sampled_rows_per_class': {
        name: int(len(reservoirs[name])) for name in class_names
    },
    'score_semantics': 'model scores are not guarantees of correctness and are not calibrated unless separately validated',
    'artifacts': artifact_files,
    'evaluation_metrics_file': 'evaluation_metrics.json',
}
write_json(EXPORT_ROOT / 'manifest.json', manifest)

BUNDLE_PATH = Path('/content/aegis_artifact_bundle.zip')
with zipfile.ZipFile(BUNDLE_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for artifact in sorted(EXPORT_ROOT.rglob('*')):
        if artifact.is_file():
            bundle.write(artifact, arcname=artifact.relative_to(EXPORT_ROOT).as_posix())

print('Artifact bundle:', BUNDLE_PATH)
print('Bundle files:', sorted(artifact_files.values()) + ['manifest.json'])
print('Measured evaluation metrics:', json.dumps(metrics, indent=2))
from google.colab import files
files.download(str(BUNDLE_PATH))